In [ ]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia
!pip install rank_bm25
!pip install pypdf
!pip install langchain_community
!pip install bitsandbytes
!pip install accelerate

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader, DirectoryLoader, WebBaseLoader
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter,CharacterTextSplitter, Language, SemanticChunker
from langchain_community.retrievers import WikipediaRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from transformers import ( AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, )
from langchain import HuggingFacePipeline
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chains import RetrievalQA
import os
import re
import torch
import pandas as pd
import numpy as np

load_dotenv()
model = ChatOpenAI()
prompt = PromptTemplate(
  template='Write a summary for the following poem - \n {poem}',
  input_variables=['poem']
)

parser = StrOutputParser()
loader = TextLoader('cricket.txt', encoding='utf-8')
docs = loader.load()
print(type(docs))
print(len(docs))
print(docs[0].page_content)
print(docs[0].metadata)

chain = prompt | model | parser
print(chain.invoke({'poem':docs[0].page_content}))

In [ ]:
loader = PyPDFLoader('dl-curriculum.pdf')
docs = loader.load()
print(type(docs))
print(len(docs))
print(docs[0].page_content)
print(docs[1].metadata)

In [ ]:
loader = CSVLoader(file_path='Social_Network_Ads.csv')
docs = loader.load()
print(len(docs))
print(docs[1])

In [ ]:
loader = DirectoryLoader(
  path='books',
  glob='*.pdf',
  loader_cls=PyPDFLoader
)

docs = loader.lazy_load()
for document in docs:
  print(document.metadata)

In [ ]:
model = ChatOpenAI()
prompt = PromptTemplate(
    template='Answer the following question \n {question} from the following text - \n {text}',
    input_variables=['question','text']
)

parser = StrOutputParser()
url='https://www.flipkart.com/apple-macbook-air-m2-16-gb-256-gb-ssd-macos-sequoia-mc7x4hn-a/p/itmdc5308fa78421'
loader=WebBaseLoader(url)
docs=loader.load()
chain = prompt | model | parser
print(chain.invoke({'question':'What is the prodcut that we are talking about?', 'text':docs[0].page_content}))

In [ ]:
text = """
Space exploration has led to incredible scientific discoveries. From landing on the Moon to exploring Mars, humanity continues to push the boundaries of what’s possible beyond our planet.

These missions have not only expanded our knowledge of the universe but have also contributed to advancements in technology here on Earth. Satellite communications, GPS, and even certain medical imaging techniques trace their roots back to innovations driven by space programs.
"""

splitter = RecursiveCharacterTextSplitter(
  chunk_size=500,
  chunk_overlap=0,
)

chunks = splitter.split_text(text)
print(len(chunks))
print(chunks)

In [ ]:
loader = PyPDFLoader('')
docs = loader.load()
splitter = CharacterTextSplitter(
  chunk_size=200,
  chunk_overlap=0,
  separator=''
)

chunks = splitter.split_documents(docs)
print(result[1].page_content)

In [ ]:
text = """
# Project Name: Smart Student Tracker

A simple Python-based project to manage and track student data, including their grades, age, and academic status.


## Features

- Add new students with relevant info
- View student details
- Check if a student is passing
- Easily extendable class-based design


## 🛠 Tech Stack

- Python 3.10+
- No external dependencies


## Getting Started

1. Clone the repo
   ```bash
   git clone https://github.com/your-username/student-tracker.git

"""

splitter = RecursiveCharacterTextSplitter.from_language(
  language=Language.MARKDOWN,
  chunk_size=200,
  chunk_overlap=0,
)

chunks = splitter.split_text(text)
print(len(chunks))
print(chunks[0])

In [ ]:
text = """
class Student:
    def __init__(self, name, age, grade):
        self.name = name
        self.age = age
        self.grade = grade  # Grade is a float (like 8.5 or 9.2)

    def get_details(self):
        return self.name"

    def is_passing(self):
        return self.grade >= 6.0


# Example usage
student1 = Student("Aarav", 20, 8.2)
print(student1.get_details())

if student1.is_passing():
    print("The student is passing.")
else:
    print("The student is not passing.")

"""

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=300,
    chunk_overlap=0,
)

chunks = splitter.split_text(text)
print(len(chunks))
print(chunks[1])

In [ ]:
text_splitter = SemanticChunker(
  OpenAIEmbeddings(),
  breakpoint_threshold_type="standard-deviation",
  breakpoint_threshold_amount=3,
)

sample = """
Farmers were working hard in the fields, preparing the soil and planting seeds for the next season. The sun was bright, and the air smelled of earth and fresh grass. The Indian Premier League (IPL) is the biggest cricket league in the world. People all over the world watch the matches and cheer for their favourite teams.


Terrorism is a big danger to peace and safety. It causes harm to people and creates fear in cities and villages. When such attacks happen, they leave behind pain and sadness. To fight terrorism, we need strong laws, alert security forces, and support from people who care about peace and safety.
"""

docs = text_splitter.create_documents([sample])
print(len(docs))
print(docs)

In [ ]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

In [ ]:
retriever = WikipediaRetriever(top_k_result=2, lang="en")
query = "the geopolitical history of india and pakistan from the perspective of a chinese"
docs = retriever.invoke(query)
docs

In [ ]:
for i, doc in enumerate(docs):
  print(f"\n--- Result {i+1} ---")
  print(f"Content:\n{doc.page_content}...")

In [ ]:
documents = [
  Document(page_content="LangChain helps developers build LLM applications easily."),
  Document(page_content="Chroma is a vector database optimized for LLM-based search."),
  Document(page_content="Embeddings convert text into high-dimensional vectors."),
  Document(page_content="OpenAI provides powerful embedding models."),
]

embedding_model = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
query = "What is Chroma used for?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)

In [ ]:
results = vectorstore.similarity_search(query=query, k=2)
for i, doc in enumerate(results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)

In [ ]:
docs = [
  Document(page_content="LangChain makes it easy to work with LLMs."),
  Document(page_content="LangChain is used to build LLM based applications."),
  Document(page_content="Chroma is used to store and search document embeddings."),
  Document(page_content="Embeddings are vector representations of text."),
  Document(page_content="MMR helps you get diverse results when doing similarity search."),
  Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(
  documents=docs,
  embedding=embedding_model,
)

retriever = vectorstore.as_retriever(
  search_type="mmr",
  search_kwargs={"k": 3, "fetch_k": 0.5},
)

query = "What is langchain?"
results = retriever.invoke(query)
for i, doc in enumerate(results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)

In [ ]:
all_docs = [
  Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
  Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
  Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
  Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
  Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
  Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
  Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
  Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
  Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
  Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
multiquery_retriever = MultiQueryRetriever.from_llm(
  retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
  llm=ChatOpenAI(model="gpt-3.5-turbo"),
)

query = "How to improve energy levels and maintain balance?"
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

for i, doc in enumerate(similarity_results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)

In [ ]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents=docs, embedding=embedding_model)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(model="gpt-3.5-turbo")
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)
for i, doc in enumerate(compressed_results):
  print(f"\n--- Result {i+1} ---")
  print(doc.page_content)

### **Hybride Search**

In [ ]:
documents = [
    "This is a list which containig sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings."
]


query="keyword-based search"
def preprocess_text(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  return text

preprocessed_documents = [preprocess_text(doc) for doc in documents]
preprocessed_documents

In [ ]:
print("Preprocessed Documents:")
for doc in preprocessed_documents:
  print(doc)

print("Preprocessed Query:")
print(query)

In [ ]:
preprocessed_query = preprocess_text(query)
vector=TfidfVectorizer()
X=vector.fit_transform(preprocessed_documents)

preprocessed_query, X.toarray(), X.toarray()[0]

In [ ]:
query_embedding=vector.transform([preprocessed_query])
similarities = cosine_similarity(X, query_embedding)
np.argsort(similarities,axis=0)

In [ ]:
ranked_indices=np.argsort(similarities,axis=0)[::-1].flatten()
ranked_documents = [documents[i] for i in ranked_indices]
for i, doc in enumerate(ranked_documents):
  print(f"Rank {i+1}: {doc}")

query, ranked_indices

In [ ]:
documents = [
    "This is a list which containig sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings."
]

document_embeddings = np.array([
    [0.634, 0.234, 0.867, 0.042, 0.249],
    [0.123, 0.456, 0.789, 0.321, 0.654],
    [0.987, 0.654, 0.321, 0.123, 0.456]
])

query_embedding = np.array([[0.789, 0.321, 0.654, 0.987, 0.123]])
similarities = cosine_similarity(document_embeddings, query_embedding)
ranked_indices = np.argsort(similarities, axis=0)[::-1].flatten()

for i, idx in enumerate(ranked_indices):
  print(f"Rank {i+1}: Document {idx+1}")

print(similarities)
print(ranked_indices)

In [ ]:
doc_path="/content/Retrieval-Augmented-Generation-for-NLP.pdf"
loader=PyPDFLoader(doc_path)
docs=loader.load()
splitter = RecursiveCharacterTextSplitter(
  chunk_size=200,
  chunk_overlap=0,
)
chunks = splitter.split_documents(docs)
chunks

In [ ]:
HF_TOKEN=""
embeddings = HuggingFaceInferenceAPIEmbeddings(api_key=HF_TOKEN, model_name="BAAI/bge-base-en-v1.5")
vectorstore=Chroma.from_documents(chunks, embeddings)
vectorstore_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 3
ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore_retriever,keyword_retriever],weights=[0.3, 0.7])

vectorstore_retriever

In [ ]:
model_name = "HuggingFaceH4/zephyr-7b-beta"

def load_quantized_model(model_name: str):
  bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
  )

  model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
  )

  return model

def initialize_tokenizer(model_name: str):
  tokenizer = AutoTokenizer.from_pretrained(model_name, return_token_type_ids=False)
  tokenizer.bos_token_id = 1
  return tokenizer


tokenizer = initialize_tokenizer(model_name)
model = load_quantized_model(model_name)

In [ ]:
pipeline=pipeline(
    "text_generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    max_length=2048,
    device_map="auto",
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=pipeline)
normal_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_retriever,
)

hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever,
)

response1 = normal_chain.invoke("What is Abstractive Question Answering?")
response1
print(response1.get("result"))

response2 = hybrid_chain.invoke("What is Abstractive Question Answering?")
response2
print(response2.get("result"))